# Kaggle SPICE Self-Play Training

This notebook implements the **SPICE (Self-Play In Corpus Environments)** framework for OnCallEnv Red Shift. It trains a single model to act as both an **LLM Attacker** (generating hard scenarios) and an **LLM Defender** (solving them) using **DrGRPO**.

## 1. GPU Check

Expected: Two Tesla T4 GPUs.

In [1]:
!nvidia-smi

Sat Apr 25 08:49:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Bootstrap Repository

Clones or updates the repository in `/kaggle/working`.

In [2]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "round2-redshift"
WORKDIR = Path("/kaggle/working/MetaHackathon-R2")

os.chdir("/kaggle/working")
if (WORKDIR / ".git").exists():
    os.chdir(WORKDIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-5"], check=True)

Cloning into '/kaggle/working/MetaHackathon-R2'...


cwd: /kaggle/working/MetaHackathon-R2
5de5a68 docs: summarize implementation and results
e3d8070 training: support interrupted grpo summaries
dcff16c updated notebook4
5ec7173 fix: keep unsloth grpo lora trainable
091fc35 fix: make kaggle notebook bootstrap repo safely


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

## 3. Install Dependencies

Installs standard requirements and LLM training stack (Unsloth, TRL).

In [3]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt
!python -m pip install pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.8 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 17.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [openenv-core] [fastmcp]]ydantic]
INFO: pip is looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of unsloth to det

## 4. Global Environment Setup

In [4]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
import sys
sys.path.append("src")
sys.path.append("scripts")

/kaggle/working
env: PYTHONPATH=src:scripts


## 5. Verify Attacker Logic

Run the unit tests to ensure the discrete action parser and variance rewards are correct on this kernel.

In [5]:
!python -m pytest tests/test_llm_attacker.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/MetaHackathon-R2
configfile: pyproject.toml
plugins: anyio-4.12.1, langsmith-0.7.6, typeguard-4.5.1
collected 0 items                                                              

============================ no tests ran in 0.00s =============================
ERROR: file or directory not found: tests/test_llm_attacker.py



## 6. SPICE Self-Play Smoke Run

Tests the full interaction loop between Attacker and Defender with minimal steps.

In [ ]:
!python scripts/train_spice_selfplay.py \
    --selfplay-iterations 2 \
    --group-size 2 \
    --batch-size 2 \
    --max-steps 5 \
    --out-dir "training_results/spice_smoke" \
    --report-to "none"

python3: can't open file '/kaggle/working/MetaHackathon-R2/scripts/train_spice_selfplay.py': [Errno 2] No such file or directory


## 7. Main SPICE Self-Play Training

Runs the primary co-evolution loop. Adjust `--report-to` to `wandb` if you have it configured.

In [ ]:
!python scripts/train_spice_selfplay.py \
    --model-name "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" \
    --selfplay-iterations 150 \
    --group-size 4 \
    --batch-size 8 \
    --max-steps 600 \
    --per-device-train-batch-size 2 \
    --gradient-accumulation-steps 4 \
    --lr 5e-6 \
    --report-to "tensorboard" \
    --out-dir "training_results/spice_selfplay_main"

## 8. Summary & Results

In [ ]:
!cat training_results/spice_selfplay_main/summary.json